In [2]:
%pip install peft evaluate soundfile
%pip install -U torchao

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

os.environ["HF_DATASETS_USE_TORCHCODEC"] = "0"

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
except:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print(e)

In [ ]:
# from huggingface_hub import sync_bucket

# sync_bucket(
#     "hf://buckets/RobChio/swissgerman",
#     "./bucket/"
# )

In [4]:
labels = {
    0: "AG",
    1: "BE",
    2: "BS",
    3: "GR",
    4: "LU",
    5: "SG",
    6: "VS",
    7: "ZH",
}

id2label = labels
label2id = {v: k for k, v in labels.items()}
num_labels = len(labels)

In [13]:
from transformers import WhisperForConditionalGeneration, WhisperForAudioClassification, WhisperProcessor
import torch

model_id = "Flix-AI/flix-swissgerman-full"

processor = WhisperProcessor.from_pretrained(model_id)

model = WhisperForAudioClassification.from_pretrained(
    model_id,
    num_labels=8,
    label2id=label2id,
    id2label=id2label,
    torch_dtype=torch.bfloat16,
)

# SpecAugment
# model.config.apply_spec_augment = True
# model.config.mask_time_prob = 0.05
# model.config.mask_feature_prob = 0.05

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


processor_config.json:   0%|          | 0.00/410 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

WhisperForAudioClassification LOAD REPORT from: Flix-AI/flix-swissgerman-full
Key                                                          | Status     | 
-------------------------------------------------------------+------------+-
model.decoder.layers.{0...31}.encoder_attn.k_proj.weight     | UNEXPECTED | 
model.decoder.layers.{0...31}.encoder_attn_layer_norm.bias   | UNEXPECTED | 
model.decoder.layers.{0...31}.encoder_attn.out_proj.weight   | UNEXPECTED | 
model.decoder.layers.{0...31}.fc1.bias                       | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn.k_proj.weight        | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn.out_proj.weight      | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn_layer_norm.weight    | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn.q_proj.weight        | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn.out_proj.bias        | UNEXPECTED | 
model.decoder.layers.{0...31}.self_attn_layer_norm.bias      | UNEXPECTED |

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # target_modules=["q_proj", "v_proj"],
    # target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
    target_modules=[
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
],
    lora_dropout=0.05,
    bias="none",
    modules_to_save=["projector", "classifier"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [18]:
import numpy as np
from collections import defaultdict
from datasets import Dataset

def balance_dataset(ds: Dataset, label_col: str = "labels", seed: int = 42) -> Dataset:
    """Oversamples minority classes in a Dataset to match the majority class count."""
    labels = ds[label_col]
    if isinstance(labels, torch.Tensor):
        labels = labels.tolist()

    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[int(label)].append(idx)

    max_samples = max(len(indices) for indices in by_label.values())
    rng = np.random.default_rng(seed)

    balanced_indices = []
    for label, indices in sorted(by_label.items()):
        sampled = rng.choice(indices, size=max_samples, replace=True)
        balanced_indices.extend(sampled)

    return ds.select(balanced_indices).shuffle(seed=seed)

In [20]:
import datasets
from datasets import load_dataset, load_from_disk, Audio, interleave_datasets, concatenate_datasets, Value

# Training data: Combine SwissDial and ArchiMob and balance by region
ds_swissdial = load_dataset("RobChio/swiss-dial-preprocessed", split="train", streaming=False)
ds_archimob_train = load_dataset("RobChio/archimob-preprocessed", split="train", streaming=False)

ds_swissdial = ds_swissdial.cast_column("dialect_code", Value(dtype="int64")) # fix type mismatch

ds_swissdial.set_format(type="torch", columns=["audio", "dialect_code"])
ds_archimob_train.set_format(type="torch", columns=["audio", "dialect_code"])

combined_train = concatenate_datasets([ds_swissdial, ds_archimob_train])
train_dataset = balance_dataset(combined_train, label_col="dialect_code", seed=42)
train_dataset = train_dataset.cast_column("audio", Audio(decode=False))
#train_dataset = ds_archimob_train.shuffle(seed=42)
#train_dataset.set_format(type="torch", columns=["audio", "dialect_code"])

# Validation data from ArchiMob
ds_archimob_val = load_dataset("RobChio/archimob-preprocessed", split="validation", streaming=False)
val_dataset = ds_archimob_val.shuffle(seed=42)
val_dataset.set_format(type="torch", columns=["audio", "dialect_code"])
val_dataset = val_dataset.cast_column("audio", Audio(decode=False))

In [ ]:
import soundfile as sf
import io

def decode_audio(batch):
    decoded_waveforms = []
    
    for item in batch["audio"]:
        # Handle dict format from Audio(decode=False) or raw bytes
        if isinstance(item, dict) and item.get("bytes"):
            speech, _ = sf.read(io.BytesIO(item["bytes"]), dtype="float32")
        elif isinstance(item, dict) and item.get("path"):
            speech, _ = sf.read(item["path"], dtype="float32")
        elif isinstance(item, bytes):
            speech, _ = sf.read(io.BytesIO(item), dtype="float32")
        else:
            speech, _ = sf.read(str(item), dtype="float32")
            
        decoded_waveforms.append(speech)

    return {"waveform": decoded_waveforms}

train_dataset_decoded = train_dataset.map(
    decode_audio,
    batched=True,
    batch_size=100,
    num_proc=2,
    remove_columns=["audio"]
)

val_dataset_decoded = val_dataset.map(
    decode_audio,
    batched=True,
    batch_size=100,
    num_proc=2,
    remove_columns=["audio"]
)

# train_dataset_decoded.save_to_disk("./bucket/train_dataset")
# val_dataset_decoded.save_to_disk("./bucket/val_dataset")

In [17]:
class FastWhisperDataCollator:
    def __init__(self, processor: WhisperProcessor, sampling_rate=16000):
        self.processor = processor
        self.sampling_rate = sampling_rate

    def __call__(self, features):
        audios = []
        for f in features:
            # Flatten and enforce 1D float32 numpy array shape (N,)
            speech = np.asarray(f["waveform"], dtype=np.float32).squeeze()
            
            # Guard against 2D stereo/channel dimensions if any remain
            if speech.ndim > 1:
                speech = speech.mean(axis=-1)
                
            audios.append(speech)

        # Batch feature extraction
        batch = self.processor(
            audios, 
            sampling_rate=self.sampling_rate, 
            return_tensors="pt"
        )
        
        batch["labels"] = torch.tensor([f["dialect_code"] for f in features], dtype=torch.long)
        return batch

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
import torch.nn.functional as F

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    
    macro_f1 = f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": macro_f1, "accuracy": acc}

training_args = TrainingArguments(
    output_dir="./swissgerman-dialect-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    #max_steps=120, # for SwissDial streaming 11213 / 256 * 3 epochs ## its actually 27828 and 3093
    num_train_epochs=3,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1", #"accuracy"
    greater_is_better=True,
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0, # multiple workers with streaming causes data duplication
    torch_compile=False, #True,
    dataloader_drop_last=False,
    push_to_hub=True,
    hub_model_id="RobChio/swissgerman-dialect-classifier",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_decoded,
    eval_dataset=val_dataset_decoded,
    data_collator=FastWhisperDataCollator(processor),
    compute_metrics=compute_metrics,
)

In [ ]:
# Train model
train_result = trainer.train()

# Log and save training state and metrics
# trainer.log_metrics("train", train_result.metrics)
# trainer.save_metrics("train", train_result.metrics)
# trainer.save_state()

In [ ]:
trainer.push_to_hub(commit_message="Finished training")

In [ ]:
# eval_metrics = trainer.evaluate(eval_dataset=test_dataset)
# trainer.log_metrics("eval", eval_metrics)
# trainer.save_metrics("eval", eval_metrics)

In [ ]:
# from google.colab import runtime
# runtime.unassign()